# SE3b_daily_prediction

**Author:** MZH Taki, Water, University of Oulu  

---

## Purpose
This notebook is **Step 3b** of the pipeline — run **daily** to generate the near-real-time prediction.

| Step | What |
|------|------|
| Cell 3 | Load calibrated parameters from `optimized_parameters.csv` (SE3a output) |
| Cell 4 | Run SWAT+ on Folder B from SE2 (observations + Harmonie forecast) |
| Cell 5 | Extract flow, TN, TP from `channel_sd_day.txt` |
| Cell 6 | Save `predictions.csv` for SE4 |

**Pre-requisites:**
- SE1 has been run today (df_obs and df_forecast downloaded)
- SE2 has been run today (Folder B created with today's forcing)
- SE3a has been run at least once (`optimized_parameters.csv` exists)

## Input
- `TxtInOut_Full_Execution_YYYY-MM-DD/` — from SE2
- `optimized_parameters.csv` — from SE3a

## Output
- `predictions.csv` — daily flow, TN, TP for the full simulation period including the forecast window

## Cell 1 — Load dependencies

In [1]:
# Modify cell
from pathlib import Path
import os
import glob
import datetime
import time
import pandas as pd
import numpy as np
import pySWATPlus

print("Dependencies loaded.")

Dependencies loaded.


## Cell 2 — Directory setup

In [2]:
# Automatically generated cell — resolve external directories
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent
external_base = project_root.parent

proc_data_dir = external_base / "oulanka_swatplus_processeddata"
model_dir     = proc_data_dir / "model"
pred_dir      = proc_data_dir / "predictions"

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
BASE_DIR    = str(external_base / "oulanka_swatplus_rawdata" / "zenodo_data")  # <-- update if needed
TARGET_UNIT = 891    # SWAT+ channel unit ID for Oulanka main outlet

# Simulation start date — must match the start of climate data in Folder B
SIM_START = '01-Jan-2015'   # <-- update if needed

# Locate today's Folder B produced by SE2
today_str   = datetime.datetime.now().strftime('%Y-%m-%d')
exec_folder = os.path.join(BASE_DIR, f"TxtInOut_Full_Execution_{today_str}")

if not os.path.exists(exec_folder):
    raise FileNotFoundError(
        f"Execution folder not found: {exec_folder}\n"
        "Please run SE2 first to create today's TxtInOut_Full_Execution folder."
    )

# Timestamped simulation output directory (unique per run)
time_str = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
sim_dir  = os.path.join(BASE_DIR, f"prediction_run_{time_str}")

print(f"Execution folder (SE2) : {os.path.basename(exec_folder)}")
print(f"Simulation output dir  : {os.path.basename(sim_dir)}")
print(f"Predictions will be saved to: {pred_dir.resolve()}")

Execution folder (SE2) : TxtInOut_Full_Execution_2026-05-22
Simulation output dir  : prediction_run_2026-05-22_18-32-00
Predictions will be saved to: /home/jovyan/oulanka_swatplus_processeddata/predictions


## Cell 3 — Load calibrated parameters from SE3a output

Parameters are read from `optimized_parameters.csv` — never hardcoded here.
This means any update to SE3a automatically flows through to SE3b.

In [3]:
# ==========================================
# CELL 3: LOAD CALIBRATED PARAMETERS FROM DISK
# ==========================================

opt_csv = model_dir / "optimized_parameters.csv"

if not opt_csv.exists():
    raise FileNotFoundError(
        f"Optimized parameters not found: {opt_csv}\n"
        "Please run SE3a Cell 8 (calibration) first."
    )

opt_df = pd.read_csv(opt_csv)

# Build the parameter list in pySWATPlus format
calibrated_params = [
    {
        'name'       : row['parameter'],
        'change_type': row['change_type'],
        'value'      : row['optimized_value']
    }
    for _, row in opt_df.iterrows()
]

print(f"Loaded {len(calibrated_params)} calibrated parameters from: {opt_csv}")
print(opt_df[['parameter', 'change_type', 'optimized_value']].to_string(index=False))

FileNotFoundError: Optimized parameters not found: /home/jovyan/oulanka_swatplus_processeddata/model/optimized_parameters.csv
Please run SE3a Cell 8 (calibration) first.

## Cell 4 — Run near-real-time prediction

SWAT+ is run on Folder B from SE2 — which contains the full historical observation record
plus the 2–3 day Harmonie NWP forecast. Calibrated parameters are applied via pySWATPlus.

The simulation end date is set automatically from the last date in the `.pcp` file so
this cell works correctly regardless of how many forecast days are available.

In [4]:
# ==========================================
# CELL 4: NEAR-REAL-TIME PREDICTION RUN
# ==========================================

# Auto-detect simulation end date from the last line of the .pcp file in Folder B
pcp_files = glob.glob(os.path.join(exec_folder, "*.pcp"))
if pcp_files:
    with open(pcp_files[0], 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    last_parts  = lines[-1].split()
    last_date   = datetime.datetime(int(last_parts[0]), 1, 1) + datetime.timedelta(int(last_parts[1]) - 1)
    # Use one day before the last forecast date to ensure complete output
    auto_end    = (last_date - datetime.timedelta(days=1)).strftime('%d-%b-%Y')
else:
    # Fallback: yesterday
    auto_end = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%d-%b-%Y')

print(f"Simulation period: {SIM_START}  →  {auto_end}")

# Set up simulation directory from Folder B
if not os.path.exists(sim_dir):
    os.makedirs(sim_dir)

txtinout_reader = pySWATPlus.TxtinoutReader(tio_dir=exec_folder)
txtinout_reader.copy_required_files(sim_dir=sim_dir)
sim_reader = pySWATPlus.TxtinoutReader(tio_dir=sim_dir)

# Enable channel_sd output
sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=True, monthly=True, yearly=True, avann=True
)

print(f"Starting prediction run in: {os.path.basename(sim_dir)}")
t0 = time.time()

sim_reader.run_swat(
    begin_date = SIM_START,
    end_date   = auto_end,
    warmup     = 2,
    parameters = calibrated_params   # apply calibrated values
)

elapsed = time.time() - t0
mins, secs = divmod(elapsed, 60)
print(f"Prediction run complete. Runtime: {int(mins)} min {secs:.1f} s")

Simulation period: 01-Jan-2015  →  23-May-2026


TypeError: Expected exactly one executable file in the parent folder, but found none or multiple

## Cell 5 — Extract flow, TN, and TP from SWAT+ output

Total Nitrogen (TN) and Total Phosphorus (TP) are calculated by summing their
constituent fractions from `channel_sd_day.txt`.

In [5]:
# ==========================================
# CELL 5: EXTRACT FLOW, TN, TP FROM OUTPUT
# ==========================================

output_file = os.path.join(sim_dir, "channel_sd_day.txt")

if not os.path.exists(output_file):
    raise FileNotFoundError(f"SWAT+ output not found: {output_file}")

df_results = pd.read_csv(output_file, sep=r'\s+', skiprows=1)

# Identify unit column (name varies slightly between SWAT+ versions)
if 'unit' in df_results.columns:
    unit_col = 'unit'
elif 'gis_id' in df_results.columns:
    unit_col = 'gis_id'
else:
    unit_col = df_results.columns[4]   # fallback

# Filter for the Oulanka main outlet
df_outlet = df_results[df_results[unit_col] == TARGET_UNIT].copy()

# Build date column from SWAT+ year/month/day columns
df_outlet['Date'] = pd.to_datetime(
    df_outlet[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'})
)

# Sum nitrogen fractions → Total Nitrogen
n_cols = [c for c in ['orgn_out', 'no3_out', 'no2_out', 'nh3_out'] if c in df_outlet.columns]
df_outlet['TN'] = df_outlet[n_cols].sum(axis=1)

# Sum phosphorus fractions → Total Phosphorus
p_cols = [c for c in ['orgp_out', 'solp_out'] if c in df_outlet.columns]
df_outlet['TP'] = df_outlet[p_cols].sum(axis=1)

print(f"Output loaded: {len(df_outlet)} days for unit {TARGET_UNIT}")

# Print the last 3 rows — these are the forecast days
forecast_rows = df_outlet.tail(3)
print("\n--- 3-Day Forecast (last 3 rows) ---")
display_cols = ['Date', 'flo_out'] + n_cols + ['TN'] + p_cols + ['TP']
display_cols = [c for c in display_cols if c in forecast_rows.columns]
print(forecast_rows[display_cols].to_string(index=False))

FileNotFoundError: SWAT+ output not found: /home/jovyan/oulanka_swatplus_rawdata/zenodo_data/prediction_run_2026-05-22_18-32-00/channel_sd_day.txt

## Cell 6 — Save predictions.csv for SE4

The complete daily time series (flow, TN, TP) is saved to `predictions.csv`.
SE4 reads this file to generate all publication figures and the interactive dashboard.

In [ ]:
# ==========================================
# CELL 6: SAVE predictions.csv FOR SE4
# ==========================================

pred_dir.mkdir(parents=True, exist_ok=True)

# Save full time series
predictions_csv = pred_dir / "predictions.csv"
df_outlet[['Date', 'flo_out', 'TN', 'TP']].to_csv(predictions_csv, index=False)

print(f"Predictions saved : {predictions_csv}")
print(f"Period            : {df_outlet['Date'].min().date()} to {df_outlet['Date'].max().date()}")
print(f"Total rows        : {len(df_outlet)}")
print("\nSE3b complete. Run SE4 to generate figures and the forecast dashboard.")

## Summary of outputs

| File | Location | Description |
|------|----------|-------------|
| `predictions.csv` | `predictions/` | Daily flow (m³/s), TN (kg), TP (kg) for full period |

SE4 reads `predictions.csv` to generate all figures and the interactive forecast dashboard.

---
## Need help?
https://github.com/orgs/DigitalWaters-fi/discussions — Tag **#modelling**